# TRM Latent-History Sudoku Study

Reproducible Colab/T4 workflow for B0/B1/B2/B3/P1. The 55-minute setting is a runtime cap, not a guarantee of convergence or completion. Scaled-model inference is useful for iteration but is not publication-scale evidence.

In [ ]:
# Runtime → T4 GPU. Always clone the study branch (default main does not have it).
import os, pathlib, sys

REPO_URL = "https://github.com/Iliyabr/trm-latent-history-attention.git"
BRANCH = "feature/latent-history-attention"
ROOT = pathlib.Path("/content/trm-latent-history-attention")

if pathlib.Path("/content").exists():
    %cd -q /content

if not (ROOT / "pretrain.py").exists():
    !git clone -b {BRANCH} {REPO_URL} trm-latent-history-attention

%cd -q {ROOT}
!git fetch origin
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}

# Do NOT pip install requirements.txt (adam-atan2/triton fail on Colab Python 3.13).
!python -m pip install -q -r requirements-colab.txt
print("cwd", pathlib.Path.cwd())
assert pathlib.Path("pretrain.py").exists(), "Setup did not land in the repo root"

## Build deterministic data

Stay in the repo root (`/content/trm-latent-history-attention`). The next cell copies `data/` from Drive if you already saved it, otherwise it builds 900 train bases (64 augs), 100 dev, and 1000 test. If the builder fails, the traceback is from that script — not from missing JSON.

In [ ]:
# Run from the repo root. Do not `%cd trm-latent-history-attention` here:
# if you are already inside the clone, that looks for a nested folder and the
# builder never writes artifacts/ under the path this cell reads.
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path("/content/trm-latent-history-attention")
if not (ROOT / "pretrain.py").exists():
    ROOT = Path.cwd()
assert (ROOT / "pretrain.py").exists(), f"Not in the study repo: {ROOT}"
os.chdir(ROOT)

DRIVE = Path("/content/drive/MyDrive/trm-study")
if DRIVE.exists():
    if (DRIVE / "data/sudoku-study-v1").exists() and not (ROOT / "data/sudoku-study-v1").exists():
        !cp -a /content/drive/MyDrive/trm-study/data ./
    if (DRIVE / "artifacts").exists() and not (ROOT / "artifacts/data").exists():
        !cp -a /content/drive/MyDrive/trm-study/artifacts ./
    if (DRIVE / "outputs").exists() and not (ROOT / "outputs/study").exists():
        !cp -a /content/drive/MyDrive/trm-study/outputs ./

manifest = ROOT / "artifacts/data/sudoku_study_v1_manifest.json"
if not manifest.exists():
    subprocess.check_call(
        [sys.executable, str(ROOT / "dataset/build_sudoku_baseline_v2.py")],
        cwd=ROOT,
    )
if not manifest.exists():
    raise FileNotFoundError(
        f"Dataset builder did not write {manifest}. The failure is in the "
        "builder output above, not this json.load line."
    )

data = json.loads(manifest.read_text())
print("cwd", ROOT)
print(data["counts"])
print(data["leakage_assertions"])

## Run one job or the suite

Start with dry-run. Change `VARIANT` and `SEED` for one of the 15 jobs. The full suite is serial and may take roughly 15 hours at the one-hour target.

In [ ]:
# One of 15 jobs: variants B0/B1/B2/B3/P1 × seeds 0/1/2.
# --dry-run only prints the command. Remove it to train.
from pathlib import Path
import os
os.chdir("/content/trm-latent-history-attention" if Path("/content/trm-latent-history-attention/pretrain.py").exists() else Path.cwd())

VARIANT, SEED = "P1", 0
!python experiments/run_study.py single --variant {VARIANT} --seed {SEED} --dry-run
# Train (T4: skip torch.compile; it warns on bfloat16):
# !python experiments/run_study.py single --variant {VARIANT} --seed {SEED} --override compile_model=false
# All 15 jobs, serial (~15h):
# !python experiments/run_study.py suite --override compile_model=false

## Resume a capped/interrupted run

In [ ]:
from pathlib import Path
import os
os.chdir("/content/trm-latent-history-attention" if Path("/content/trm-latent-history-attention/pretrain.py").exists() else Path.cwd())

# Continues the same VARIANT/SEED from runtime_cap.pt or the latest step_*.pt.
!python experiments/run_study.py resume --variant {VARIANT} --seed {SEED} --dry-run
# !python experiments/run_study.py resume --variant {VARIANT} --seed {SEED} --override compile_model=false

## Long B0 vs P1 (one seed)

The 15-job runs stopped after ~1800 steps (~8 minutes) because **`epochs=64` finished**, not because of the 55-minute cap. Exact Sudoku accuracy stayed at 0: that metric requires every cell correct, which this schedule never reached.

This block keeps the **same D256 / H2 / L4 / ACT6 model** and only trains longer. Run **one variant per Colab session** (set `VARIANT` to `"B0"`, train until disconnect, copy `outputs/` to Drive, then a new session with `"P1"`). Outputs go to `outputs/study/colab_heavy/` so they do not overwrite the short runs.

Edit the numbered knobs, then run. `epochs` must stay divisible by `eval_interval`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path("/content/trm-latent-history-attention")
if not (ROOT / "pretrain.py").exists():
    ROOT = Path.cwd()
os.chdir(ROOT)
os.environ["PYTHONUNBUFFERED"] = "1"

VARIANT = "B0"   # later session: "P1"
SEED = 0
PRESET = "colab_heavy"

EPOCHS = 8192
EVAL_INTERVAL = 32
MAX_RUNTIME_MINUTES = 700
BATCH_SIZE = 32
COMPILE_MODEL = False

assert EPOCHS % EVAL_INTERVAL == 0
est_steps = EPOCHS * 900 // BATCH_SIZE
print(f"expected ~{est_steps} optimizer steps; tqdm is step/total")
print("epoch lines print every", EVAL_INTERVAL, "epochs; checkpoints appear then too")

cmd = [
    sys.executable, "-u", "experiments/run_study.py", "single",
    "--preset", PRESET, "--variant", VARIANT, "--seed", str(SEED),
    "--override", f"epochs={EPOCHS}",
    "--override", f"eval_interval={EVAL_INTERVAL}",
    "--override", f"max_runtime_minutes={MAX_RUNTIME_MINUTES}",
    "--override", f"global_batch_size={BATCH_SIZE}",
    "--override", f"compile_model={str(COMPILE_MODEL).lower()}",
    "--override", "lr=1e-4",
    "--override", "lr_warmup_steps=500",
]
print(" ".join(cmd), flush=True)
proc = subprocess.Popen(
    cmd,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait():
    raise SystemExit(f"Training exited {proc.returncode}. Scroll up for the real error.")

## Resume / copy the long run

If Colab disconnects, resume the **same** `VARIANT` / `SEED` / `PRESET`. Then copy `outputs/study/colab_heavy/` to Drive before the VM dies.

In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path("/content/trm-latent-history-attention")
if not (ROOT / "pretrain.py").exists():
    ROOT = Path.cwd()
os.chdir(ROOT)

# Must match the long-run cell.
VARIANT = "B0"
SEED = 0
PRESET = "colab_heavy"
RESUME = False  # True continues from runtime_cap.pt / latest step_*.pt
COPY_TO_DRIVE = True

cmd = [
    sys.executable, "experiments/run_study.py",
    "resume" if RESUME else "single",
    "--preset", PRESET, "--variant", VARIANT, "--seed", str(SEED),
]
if RESUME:
    subprocess.check_call(cmd, cwd=ROOT)

run_dir = ROOT / "outputs" / "study" / PRESET / f"{VARIANT}-seed{SEED}"
print("run_dir", run_dir)
print("checkpoints", sorted(p.name for p in run_dir.glob("*.pt")))

if COPY_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped:", exc)
    else:
        dest = Path("/content/drive/MyDrive/trm-study/outputs")
        dest.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(["cp", "-a", str(ROOT / "outputs" / "study" / PRESET), str(dest)])
        print("copied", PRESET, "to", dest)

## Evaluate long B0 vs P1 (seed 0)

Uses `outputs/study/colab_heavy/`. Writes `results/study-heavy/` and copies it to Drive. Needs `best_dev.pt` (or `runtime_cap.pt`) for both variants.

In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path("/content/trm-latent-history-attention")
if not (ROOT / "pretrain.py").exists():
    ROOT = Path.cwd()
os.chdir(ROOT)

PRESET = "colab_heavy"
SEED = 0
OUTPUT = ROOT / "results" / "study-heavy"
CONFIG = "config/experiment/sudoku_study_colab_heavy.yaml"


def pick_checkpoint(run_dir: Path) -> Path | None:
    for name in ("best_dev.pt", "runtime_cap.pt"):
        candidate = run_dir / name
        if candidate.exists():
            return candidate
    steps = sorted(run_dir.glob("step_*.pt"), key=lambda p: int(p.stem.removeprefix("step_")))
    return steps[-1] if steps else None

jobs = []
for variant in ("B0", "P1"):
    run_dir = ROOT / "outputs" / "study" / PRESET / f"{variant}-seed{SEED}"
    ckpt = pick_checkpoint(run_dir)
    print(f"{variant}-seed{SEED}: {ckpt or 'NO CHECKPOINT'}")
    if ckpt is not None:
        jobs.append((variant, ckpt))
if len(jobs) < 2:
    raise SystemExit("Need checkpoints for both B0 and P1 under outputs/study/colab_heavy/.")

for variant, ckpt in jobs:
    subprocess.check_call(
        [
            sys.executable, "experiments/evaluate_study.py",
            "--config", CONFIG,
            "--checkpoint", f"{variant}={ckpt.as_posix()}",
            "--data", "data/sudoku-study-v1",
            "--split", "test",
            "--seed", str(SEED),
            "--output", str(OUTPUT),
            "--interventions",
        ],
        cwd=ROOT,
    )

subprocess.check_call(
    [
        sys.executable, "experiments/analyze_results.py",
        "--input", str(OUTPUT),
        "--output", str(OUTPUT / "analysis"),
    ],
    cwd=ROOT,
)
print("wrote", OUTPUT / "analysis")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped:", exc)
else:
    dest = Path("/content/drive/MyDrive/trm-study/results")
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["cp", "-a", str(OUTPUT), str(dest)])
    print("copied to", dest / "study-heavy")

## Evaluate every finished job

Scans `outputs/study/colab/` for all `B0`–`B3`/`P1` × seeds `0`–`2` that have a checkpoint. Prefers `best_dev.pt`, then `runtime_cap.pt`, then the latest `step_*.pt`. Writes test metrics to `results/study/` and copies that folder to Google Drive `MyDrive/trm-study/results`. Jobs with no weights are skipped.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path("/content/trm-latent-history-attention")
if not (ROOT / "pretrain.py").exists():
    ROOT = Path.cwd()
assert (ROOT / "pretrain.py").exists(), f"Not in the study repo: {ROOT}"
os.chdir(ROOT)

VARIANTS = ("B0", "B1", "B2", "B3", "P1")
SEEDS = (0, 1, 2)
OUTPUT = ROOT / "results" / "study"
DRIVE = Path("/content/drive/MyDrive/trm-study")


def pick_checkpoint(run_dir: Path) -> Path | None:
    for name in ("best_dev.pt", "runtime_cap.pt"):
        candidate = run_dir / name
        if candidate.exists():
            return candidate
    steps = sorted(
        run_dir.glob("step_*.pt"),
        key=lambda path: int(path.stem.removeprefix("step_")),
    )
    return steps[-1] if steps else None


jobs = []
missing = []
for variant in VARIANTS:
    for seed in SEEDS:
        run_dir = ROOT / "outputs" / "study" / "colab" / f"{variant}-seed{seed}"
        checkpoint = pick_checkpoint(run_dir)
        print(f"{variant}-seed{seed}: {checkpoint or 'NO CHECKPOINT'}")
        if checkpoint is None:
            missing.append(f"{variant}-seed{seed}")
            continue
        jobs.append((variant, seed, checkpoint))

if not jobs:
    raise SystemExit("No .pt checkpoints under outputs/study/colab/. Copy outputs from Drive first.")

for variant, seed, checkpoint in jobs:
    print(f"\n=== evaluate {variant} seed {seed} ({checkpoint.name}) ===")
    subprocess.check_call(
        [
            sys.executable,
            "experiments/evaluate_study.py",
            "--config",
            "config/experiment/sudoku_study_colab.yaml",
            "--checkpoint",
            f"{variant}={checkpoint.as_posix()}",
            "--data",
            "data/sudoku-study-v1",
            "--split",
            "test",
            "--seed",
            str(seed),
            "--output",
            str(OUTPUT),
            "--interventions",
        ],
        cwd=ROOT,
    )

subprocess.check_call(
    [
        sys.executable,
        "experiments/analyze_results.py",
        "--input",
        str(OUTPUT),
        "--output",
        str(OUTPUT / "analysis"),
    ],
    cwd=ROOT,
)

if missing:
    print("skipped (no weights):", ", ".join(missing))
print("wrote", OUTPUT / "analysis")
for name in ("seed_results.csv", "aggregate_results.csv", "analysis.json"):
    path = OUTPUT / "analysis" / name
    if path.exists():
        print(path.name, "ok")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped:", exc)
else:
    dest = DRIVE / "results"
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["cp", "-a", str(OUTPUT), str(DRIVE / "results")])
    print("copied to", DRIVE / "results")